In [9]:
import pandas as pd

# ========= INPUT =========
INPUT_FILE = "deduplicated_Verified_manually.xlsx"
OUTPUT_FILE = "multimodal_screening.xlsx"

df = pd.read_excel(INPUT_FILE)

# ========= INITIALISATION =========
df["Decision"] = ""
df["Reason for Exclusion"] = ""
df["Score"] = 0.0

# ========= KEYWORDS =========
INCLUDE_KEYWORDS = [
    "vision", "visual",
    "tactile", "haptic",
    "multimodal", "fusion", "cross-modal",
    "object", "texture", "material"
]

EXCLUDE_KEYWORDS = [
    "survey", "review", "bibliography", "meta-analysis"
]

# ========= SCORE FUNCTION (0–10) =========

def compute_score(title):
    t = str(title).lower()

    include_hits = sum(k in t for k in INCLUDE_KEYWORDS)
    exclude_hits = sum(k in t for k in EXCLUDE_KEYWORDS)

    # score brut
    score = include_hits * 2.5
    score -= exclude_hits * 3

    # clamp entre 0 et 10
    score = max(0, min(10, score))

    return round(score, 2), include_hits, exclude_hits

# ========= DECISION =========

def decide(score, exclude_hits):
    if exclude_hits >= 1:
        return "EXCLUDE", "Review / Survey paper"

    if score >= 5:
        return "INCLUDE", ""

    elif score >= 2:
        return "MAYBE", "Moderate relevance"

    else:
        return "EXCLUDE", "Low relevance / out of scope"

# ========= APPLICATION =========

for i in range(len(df)):
    score, inc, exc = compute_score(df.loc[i, "Title"])

    df.loc[i, "Score"] = score

    decision, reason = decide(score, exc)

    df.loc[i, "Decision"] = decision
    df.loc[i, "Reason for Exclusion"] = reason

# ========= COMPTAGE =========

total = len(df)
include_count = (df["Decision"] == "INCLUDE").sum()
exclude_count = (df["Decision"] == "EXCLUDE").sum()
maybe_count = (df["Decision"] == "MAYBE").sum()

print("\n===== SCREENING SUMMARY =====")
print(f"Total articles : {total}")
print(f"INCLUDE : {include_count}")
print(f"EXCLUDE : {exclude_count}")
print(f"MAYBE : {maybe_count}")

print("\n===== SCORE STATS (/10) =====")
print(f"Score moyen : {df['Score'].mean():.2f}")
print(f"Score max : {df['Score'].max()}")
print(f"Score min : {df['Score'].min()}")

# ========= EXPORT =========

df.to_excel(OUTPUT_FILE, index=False)

print("\nScreening terminé ! Fichier :", OUTPUT_FILE)


===== SCREENING SUMMARY =====
Total articles : 238
INCLUDE : 105
EXCLUDE : 57
MAYBE : 76

===== SCORE STATS (/10) =====
Score moyen : 3.64
Score max : 10.0
Score min : 0.0

Screening terminé ! Fichier : multimodal_screening.xlsx
